In [ ]:
# ============ IMPORTS ============
import pandas as pd
import numpy as np
import talib

# ========== CONFIG ============
include_fgi = False  # Whether to include Fear & Greed Index as a feature
rsi_period = 14
macd_fast_period = 12
macd_slow_period = 26
macd_signal_period = 9
bollinger_bands_period = 20
adx_period = 14
aroon_period = 14
plus_di_period = 14
minus_di_period = 14
atr_period = 14
natr_period = 14
stochrsi_period = 14
stoch_period = 14
stoch_fast_k = 3
stoch_fast_d = 3
willr_period = 14
mfi_period = 14
cci_period = 20
cmo_period = 14
kama_period = 10
ultosc_period_1 = 7
ultosc_period_2 = 14
ultosc_period_3 = 28
ppo_fast_period = 12
ppo_slow_period = 26
trix_period = 15
linearreg_period = 20
sar_af = 0.02
sar_af_max = 0.2
bollinger_bands_std_deviations = 2
ewm_span = 100
adosc_fast_period = 3
adosc_slow_period = 10
# ============================

# ============ LOAD CACHED DATA ============
bitcoin_data_cache = pd.read_pickle('data_cache/01_bitcoin_data.pkl')
bitcoin_price_history = bitcoin_data_cache['bitcoin_price_history']
fear_greed_index_data = bitcoin_data_cache['fgi_data']
metrics_data = bitcoin_data_cache['metrics_data']

print(f"Loaded from {bitcoin_data_cache['cache_date']}")

In [ ]:
# ============ CONSOLIDATED DATAFRAME ============
bitcoin_price_and_features = bitcoin_price_history.copy()

# Fear & Greed Index - market sentiment indicator (0=Extreme Fear, 100=Extreme Greed)
if include_fgi:
    bitcoin_price_and_features['Fear_and_Greed_Index'] = fear_greed_index_data['value']
    
# Coin Metrics loaded metrics
for column in metrics_data.columns:
    if column != 'time':
        bitcoin_price_and_features[column] = metrics_data[column]
        
# ============ DATA IMPUTATION ============
bitcoin_price_and_features = bitcoin_price_and_features.interpolate(method='linear', limit=3, limit_direction='forward')


In [ ]:
# ============ CALCULATE ALL FEATURES ============
high = bitcoin_price_and_features['High'].values.astype(np.float64)
low = bitcoin_price_and_features['Low'].values.astype(np.float64)
close = bitcoin_price_and_features['Close'].values.astype(np.float64)
volume = bitcoin_price_and_features['Volume'].values.astype(np.float64)
        
# RSI - measures if Bitcoin is overbought (>70) or oversold (<30)
bitcoin_price_and_features['RSI'] = talib.RSI(close, timeperiod=rsi_period)

# MACD - shows momentum and trend direction
macd_line, macd_signal_line, macd_histogram = talib.MACD(
    close,
    fastperiod=macd_fast_period,
    slowperiod=macd_slow_period,
    signalperiod=macd_signal_period
)
bitcoin_price_and_features['MACD_Line'] = macd_line
bitcoin_price_and_features['MACD_Histogram'] = macd_histogram
bitcoin_price_and_features['MACD_Signal'] = macd_signal_line

# Bollinger Bands - shows price volatility and potential breakouts
upper_band, middle_band, lower_band = talib.BBANDS(
    close,
    timeperiod=bollinger_bands_period,
    nbdevup=bollinger_bands_std_deviations,
    nbdevdn=bollinger_bands_std_deviations
)
bitcoin_price_and_features['BB_Upper'] = upper_band
bitcoin_price_and_features['BB_Middle'] = middle_band
bitcoin_price_and_features['BB_Lower'] = lower_band

# Daily price changes
bitcoin_price_and_features['Daily_Return'] = bitcoin_price_and_features['Close'].pct_change()
bitcoin_price_and_features['Log_Return'] = np.log(
    bitcoin_price_and_features['Close'] / bitcoin_price_and_features['Close'].shift(1)
)

# Moving averages (trend indicators for different time periods)
moving_average_windows = [7, 14, 30, 50, 200]  # 1 week to ~7 months
for num_days in moving_average_windows:
    # Simple Moving Average (equal weight to all days)
    bitcoin_price_and_features[f'SMA_{num_days}'] = bitcoin_price_and_features['Close'].rolling(window=num_days).mean()
    # Exponential Moving Average (more weight to recent days)
    bitcoin_price_and_features[f'EMA_{num_days}'] = bitcoin_price_and_features['Close'].ewm(span=num_days).mean()

# Volatility measures (how much price fluctuates)
bitcoin_price_and_features['Volatility_7day'] = bitcoin_price_and_features['Daily_Return'].rolling(window=7).std()
bitcoin_price_and_features['Volatility_30day'] = bitcoin_price_and_features['Daily_Return'].rolling(window=30).std()
bitcoin_price_and_features['Volatility_EWMA'] = bitcoin_price_and_features['Daily_Return'].ewm(span=ewm_span).std()

# Volume indicators (trading activity level)
bitcoin_price_and_features['Volume_SMA_7day'] = bitcoin_price_and_features['Volume'].rolling(window=7).mean()
bitcoin_price_and_features['Volume_SMA_30day'] = bitcoin_price_and_features['Volume'].rolling(window=30).mean()
bitcoin_price_and_features['Volume_Daily_Change'] = bitcoin_price_and_features['Volume'].pct_change()

# Momentum indicators (rate of price change)
momentum_windows = [7, 14, 30]
for num_days in momentum_windows:
    # Absolute momentum (price difference)
    bitcoin_price_and_features[f'Momentum_{num_days}day'] = (
        bitcoin_price_and_features['Close'] - bitcoin_price_and_features['Close'].shift(num_days)
    )
    # Rate of Change (percentage change)
    bitcoin_price_and_features[f'ROC_{num_days}day'] = bitcoin_price_and_features['Close'].pct_change(periods=num_days)

# Bollinger Band Width (measures volatility)
bitcoin_price_and_features['BB_Width'] = (bitcoin_price_and_features['BB_Upper'] - bitcoin_price_and_features['BB_Lower']) / bitcoin_price_and_features['BB_Middle']

# ADX — trend strength (0-100, regardless of direction)
bitcoin_price_and_features['ADX'] = talib.ADX(high, low, close, timeperiod=adx_period)
bitcoin_price_and_features['ADXR'] = talib.ADXR(high, low, close, timeperiod=adx_period)
bitcoin_price_and_features['PLUS_DI']  = talib.PLUS_DI(high, low, close, timeperiod=plus_di_period)
bitcoin_price_and_features['MINUS_DI'] = talib.MINUS_DI(high, low, close, timeperiod=minus_di_period)
bitcoin_price_and_features['AROONOSC'] = talib.AROONOSC(high, low, timeperiod=aroon_period)

# Volatility indicators — how much price moves, regardless of direction
bitcoin_price_and_features['ATR']      = talib.ATR(high, low, close, timeperiod=atr_period)
bitcoin_price_and_features['NATR']     = talib.NATR(high, low, close, timeperiod=natr_period)
bitcoin_price_and_features['TRANGE']   = talib.TRANGE(high, low, close)
bitcoin_price_and_features['STOCHRSI'] = talib.STOCHRSI(close, timeperiod=stochrsi_period)[0]  # fastk
bitcoin_price_and_features['WILLR']    = talib.WILLR(high, low, close, timeperiod=willr_period)
bitcoin_price_and_features['OBV']      = talib.OBV(close, volume)

# Stochastic Oscillator — overbought-oversold (slower than STOCHRSI)
stoch_k, stoch_d = talib.STOCH(high, low, close, fastk_period=stoch_period, slowk_period=stoch_fast_k, slowd_period=stoch_fast_d)
bitcoin_price_and_features['STOCH_K'] = stoch_k
bitcoin_price_and_features['STOCH_D'] = stoch_d

# Money Flow Index — volume-weighted RSI-like indicator
bitcoin_price_and_features['MFI'] = talib.MFI(high, low, close, volume, timeperiod=mfi_period)

# Accumulation/Distribution Line — volume-weighted price action
bitcoin_price_and_features['AD'] = talib.AD(high, low, close, volume)

# Chaikin A/D Oscillator — short/long EMA of A/D line (volume momentum)
bitcoin_price_and_features['ADOSC'] = talib.ADOSC(high, low, close, volume, fastperiod=adosc_fast_period, slowperiod=adosc_slow_period)

# Commodity Channel Index — mean reversion / cyclical extremes
bitcoin_price_and_features['CCI'] = talib.CCI(high, low, close, timeperiod=cci_period)

# Parabolic SAR — trend reversal and stop-loss levels
bitcoin_price_and_features['SAR'] = talib.SAR(high, low, acceleration=sar_af, maximum=sar_af_max)

# Kaufman Adaptive Moving Average — noise-adaptive trend smoothing
bitcoin_price_and_features['KAMA'] = talib.KAMA(close, timeperiod=kama_period)

# Adaptive Moving Averages — fast/slow adaptive trend followers
bitcoin_price_and_features['MAMA'], bitcoin_price_and_features['FAMA'] = talib.MAMA(close)

# Ultimate Oscillator — multi-timeframe momentum (7, 14, 28 period weighted)
bitcoin_price_and_features['ULTOSC'] = talib.ULTOSC(high, low, close, timeperiod1=ultosc_period_1, timeperiod2=ultosc_period_2, timeperiod3=ultosc_period_3)

# Chande Momentum Oscillator — momentum with extended range
bitcoin_price_and_features['CMO'] = talib.CMO(close, timeperiod=cmo_period)

# Percentage Price Oscillator — normalised MACD (useful for comparing instruments)
bitcoin_price_and_features['PPO'] = talib.PPO(close, fastperiod=ppo_fast_period, slowperiod=ppo_slow_period)

# TRIX — Triple Exponential Moving Average momentum (momentum of momentum)
bitcoin_price_and_features['TRIX'] = talib.TRIX(close, timeperiod=trix_period)

# Linear Regression Slope — trend strength via linear regression coefficient
bitcoin_price_and_features['LINEARREG_SLOPE'] = talib.LINEARREG_SLOPE(close, timeperiod=linearreg_period)

In [ ]:
# ============ PREPARE FEATURE DATAFRAME ============
# Extract only engineered features (no raw OHLCV) and drop NaN
bitcoin_features = bitcoin_price_and_features.drop(['Close','High','Low','Open','Volume'], axis=1)

In [ ]:
# =========== CACHE PROCESSED DATA ============
pd.to_pickle({
    'bitcoin_price_and_features': bitcoin_price_and_features,
    'bitcoin_features': bitcoin_features
}, 'data_cache/02_engineered_features.pkl')